# Agentic Design Patterns in LangGraph: Orchestration and Evaluation

This notebook implements two agentic workflow patterns with current LangGraph and LangChain APIs:

- **Orchestrator-worker**: an orchestrator decomposes a request into runtime-generated tasks, workers process those tasks in parallel, and a synthesizer combines the results.
- **Evaluator-optimizer**: a generator creates a draft, an evaluator returns structured feedback, and a conditional edge loops until the result is accepted or the iteration limit is reached.

## Setup

The notebook expects your OpenAI credentials and related settings in `.env`. Dependencies are managed by the repository environment, not by pinned `%pip install` cells.

In [ ]:
import operator
from typing import Annotated, Literal, TypedDict

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langgraph.graph import START, END, StateGraph
from langgraph.types import Send
from pydantic import BaseModel, Field

load_dotenv()

llm = init_chat_model(model="gpt-5.4", model_provider="openai", temperature=0)

In [ ]:
def show_graph(app):
    """Display a Mermaid graph when notebook rendering support is available."""
    try:
        from IPython.display import Image, display

        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception as exc:
        print(f"Graph visualization skipped: {exc}")

## Completed Exercise 1: Orchestrator-Worker Pattern

The orchestrator-worker graph below builds a meal planning assistant:

1. The orchestrator turns a free-form meal request into structured dish tasks.
2. `assign_workers(...)` returns one `Send(...)` object per dish, creating runtime fan-out.
3. Each chef worker receives branch-specific state for one dish.
4. Worker outputs are merged with `Annotated[list[str], operator.add]`.
5. The synthesizer combines all worker results into the final guide.

In [ ]:
class DishTask(BaseModel):
    name: str = Field(description="Name of the dish to prepare.")
    cuisine: str = Field(description="Cuisine or cultural origin of the dish.")
    ingredients: list[str] = Field(description="Ingredients needed for the dish.")


class MealPlan(BaseModel):
    dishes: list[DishTask] = Field(
        description="Independent dish tasks that can be assigned to chef workers."
    )


class MealState(TypedDict):
    request: str
    dishes: list[dict]
    completed_menu: Annotated[list[str], operator.add]
    final_meal_guide: str


class ChefWorkerState(TypedDict):
    dish: dict


meal_planner_llm = llm.with_structured_output(MealPlan)

In [ ]:
def orchestrator(state: MealState) -> dict:
    plan = meal_planner_llm.invoke(
        "Break this meal request into independent dishes. For each dish, "
        "include a cuisine and ingredient list.\n\n"
        f"Meal request: {state['request']}"
    )
    return {"dishes": [dish.model_dump() for dish in plan.dishes]}


def assign_workers(state: MealState) -> list[Send]:
    return [Send("chef_worker", {"dish": dish}) for dish in state["dishes"]]


def chef_worker(state: ChefWorkerState) -> dict:
    dish = state["dish"]
    response = llm.invoke(
        f"""
        You are a world-class chef specializing in {dish['cuisine']} cuisine.
        Create a practical cooking guide for this dish.

        Dish: {dish['name']}
        Ingredients: {', '.join(dish['ingredients'])}

        Include preparation steps, cooking guidance, and serving notes.
        """
    )
    return {"completed_menu": [response.content.strip()]}


def synthesize_menu(state: MealState) -> dict:
    guide = "\n\n---\n\n".join(state["completed_menu"])
    return {"final_meal_guide": guide}

In [ ]:
meal_graph = StateGraph(MealState)
meal_graph.add_node("orchestrator", orchestrator)
meal_graph.add_node("chef_worker", chef_worker)
meal_graph.add_node("synthesizer", synthesize_menu)

meal_graph.add_edge(START, "orchestrator")
meal_graph.add_conditional_edges("orchestrator", assign_workers)
meal_graph.add_edge("chef_worker", "synthesizer")
meal_graph.add_edge("synthesizer", END)

meal_app = meal_graph.compile()
show_graph(meal_app)

In [ ]:
meal_result = meal_app.invoke({
    "request": "Plan a dinner with spaghetti bolognese, chicken stir fry, and carrot cake."
})

print(meal_result["final_meal_guide"][:2000])

## Completed Exercise 2: Evaluator-Optimizer Pattern

The evaluator-optimizer graph below builds an iterative investment-plan workflow:

1. A setup node classifies the investor profile into a target risk grade using structured output.
2. A generator creates an initial plan, then revises it when feedback exists.
3. An evaluator returns a structured grade and feedback.
4. `add_conditional_edges(...)` accepts the plan or loops back to the generator.
5. The loop stops when the grade matches the target or the iteration limit is reached.

In [ ]:
RiskGrade = Literal[
    "ultra_conservative",
    "conservative",
    "moderate",
    "growth",
    "high_risk",
]


class InvestmentState(TypedDict):
    investor_profile: str
    target_grade: RiskGrade
    investment_plan: str
    current_grade: RiskGrade
    feedback: str
    iterations: int


class RiskProfile(BaseModel):
    grade: RiskGrade = Field(description="Target risk grade for the investor profile.")


class InvestmentEvaluation(BaseModel):
    grade: RiskGrade = Field(description="Risk grade assigned to the investment plan.")
    feedback: str = Field(description="Specific feedback for improving risk alignment.")


MAX_ITERATIONS = 3
risk_profile_llm = llm.with_structured_output(RiskProfile)
investment_evaluator_llm = llm.with_structured_output(InvestmentEvaluation)

In [ ]:
def determine_target_grade(state: InvestmentState) -> dict:
    result = risk_profile_llm.invoke(
        "Classify this investor profile into one risk grade: "
        "ultra_conservative, conservative, moderate, growth, or high_risk.\n\n"
        f"Investor profile: {state['investor_profile']}"
    )
    return {"target_grade": result.grade}


def generate_investment_plan(state: InvestmentState) -> dict:
    if state.get("feedback"):
        prompt = f"""
        You are a diversified, risk-aware investment strategist. Revise the plan
        using the evaluator feedback while targeting this risk grade: {state['target_grade']}.

        Investor profile:
        {state['investor_profile']}

        Previous plan:
        {state['investment_plan']}

        Evaluator feedback:
        {state['feedback']}
        """
    else:
        prompt = f"""
        You are a growth-oriented investment strategist. Create an initial plan
        for the investor below while respecting the target risk grade: {state['target_grade']}.

        Investor profile:
        {state['investor_profile']}
        """

    response = llm.invoke(prompt)
    return {"investment_plan": response.content.strip()}


def evaluate_investment_plan(state: InvestmentState) -> dict:
    result = investment_evaluator_llm.invoke(
        f"""
        Evaluate this investment plan against the investor profile and target risk grade.
        Return a risk grade and concrete feedback.

        Investor profile:
        {state['investor_profile']}

        Target risk grade:
        {state['target_grade']}

        Investment plan:
        {state['investment_plan']}
        """
    )
    return {
        "current_grade": result.grade,
        "feedback": result.feedback,
        "iterations": state.get("iterations", 0) + 1,
    }


def route_investment(state: InvestmentState) -> Literal["accepted", "revise"]:
    if state["current_grade"] == state["target_grade"]:
        return "accepted"
    if state["iterations"] >= MAX_ITERATIONS:
        return "accepted"
    return "revise"

In [ ]:
optimizer_graph = StateGraph(InvestmentState)
optimizer_graph.add_node("determine_target_grade", determine_target_grade)
optimizer_graph.add_node("generate_plan", generate_investment_plan)
optimizer_graph.add_node("evaluate_plan", evaluate_investment_plan)

optimizer_graph.add_edge(START, "determine_target_grade")
optimizer_graph.add_edge("determine_target_grade", "generate_plan")
optimizer_graph.add_edge("generate_plan", "evaluate_plan")
optimizer_graph.add_conditional_edges(
    "evaluate_plan",
    route_investment,
    {
        "accepted": END,
        "revise": "generate_plan",
    },
)

optimizer_app = optimizer_graph.compile()
show_graph(optimizer_app)

In [ ]:
investment_result = optimizer_app.invoke({
    "investor_profile": (
        "Age: 29\n"
        "Salary: $110,000\n"
        "Assets: $40,000\n"
        "Goal: Achieve financial independence by age 45\n"
        "Risk tolerance: High"
    ),
    "iterations": 0,
})

print("Target grade:", investment_result["target_grade"])
print("Final grade:", investment_result["current_grade"])
print("Iterations:", investment_result["iterations"])
print("\nFeedback:\n", investment_result["feedback"])
print("\nFinal investment plan:\n", investment_result["investment_plan"])

## Conclusion

You now have two completed LangGraph agentic design patterns:

- Orchestrator-worker for dynamic task decomposition and parallel worker execution.
- Evaluator-optimizer for iterative generation, structured evaluation, and conditional refinement.

## Authors

[Joseph Santarcangelo](https://author.skills.network/instructors/joseph_santarcangelo) | Data Scientist @ IBM <br>
[Wojciech "Victor" Fulmyk](https://author.skills.network/instructors/wojciech_fulmyk) | Data Scientist @ IBM <br>
[Kunal Makwana](https://author.skills.network/instructors/kunal_makwana) | AI Software Developer @ IBM<br>
[Joshua Zhou](https://author.skills.network/instructors/joshua_zhou) | Data Scientist Intern @ IBM
